Structured output <break>
<break>
Models can be requested to provide their responses in a format matching a given schema


In [1]:
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [4]:
os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

In [5]:
model = init_chat_model('groq:qwen/qwen3-32b')

### Pydantic<break>
pydantic models provide the richest feature set with field validation, descriptions, and nested structures

In [6]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description="The title ofv the movie")
    year:int = Field(description="This year the movie was released")
    director: str = Field(description="The director of the movie")
    ratings: float = Field(description="The movie's ratings")

In [7]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002838962A330>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002838AC72ED0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title ofv the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'ratings': {'description': "The movie's ratings

In [8]:
model_with_structure.invoke("Provide me the details about the movie Vanilla sky")

Movie(title='Vanilla Sky', year=2001, director='Cameron Crowe', ratings=7.5)

In [ ]:
class Biography(BaseModel):
    name:str = Field(description="Name of the person")
    age:int = Field(description="Age of the person")
    nationality:str = Field(description="Nationality of the person")
    father_name: str = Field(description="Father's name")
    mother_name: str = Field(description="Mother's name")
    profession: str = Field(description="Profession of the person")

In [13]:
model_with_bio_structure = model.with_structured_output(Biography)

In [15]:
model_with_bio_structure.invoke("Who is salman khan?")

Biography(name='Salman Khan', age=58, nationality='Indian', father_name='Salim Khan', mother_name='Saba Khan', profession='Actor, producer, and philanthropist')

Message output along with parsed structure

In [19]:
class Biography(BaseModel):
    name:str = Field("Name of the person")
    age:int = Field("Age of the person")
    nationality:str = Field("Nationality of the person")
    father_name: str = Field("Father's name")
    mother_name: str = Field("Mother's name")
    profession: str = Field("Profession of the person")

model_with_bio_structure = model.with_structured_output(Biography, include_raw=True)


In [20]:
model_with_bio_structure.invoke("Who is salman khan?")

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking "Who is Salman Khan?" I need to provide a biography of him. Let me recall the available function. The Biography function has parameters like name, age, father_name, mother_name, nationality, and profession. \n\nFirst, I know Salman Khan is a well-known Indian actor. His full name is Salman Khan. His age might be around 55, but I should verify that. His father is Salim Khan, and his mother is Sushila Khan. He\'s been in the film industry for a long time, so his profession is actor and producer. Nationality is Indian. \n\nWait, the age parameter is an integer. Let me check his birth year. He was born on December 27, 1965. So as of 2023, he\'s 57 years old. But since the user might be asking at any time, maybe I should use the current year minus 1965. But since the function expects a default value, maybe just input 58? Or does the function calculate it? The default is "Age of the person", but I

### Nested structure

In [24]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    age: int

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genred: list[str]
    budget: float | None = Field(None, description="Budget of the movie in USD")

model_wih_movie_structure = model.with_structured_output(MovieDetails)

In [27]:
response = model_wih_movie_structure.invoke("Vanilla sky")
print(response)

title='Vanilla Sky' year=2001 cast=[Actor(name='Tom Cruise', age=37), Actor(name='Cameron Diaz', age=28), Actor(name='Jason Lee', age=30)] genred=['Drama', 'Science Fiction'] budget=None


### TypeDict

In [33]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    title: Annotated[str,"title of the movie"]
    year: Annotated[int, "release year of the movie"]
    director: Annotated[str, "director of the movie"]
    rating: Annotated[float,"rating of the movie on imdb"]

model_with_type_dict = model.with_structured_output(MovieDict)
response = model_with_type_dict.invoke("tell me the details of avengers endgame")
print(response)

{'director': 'Joe Russo, Anthony Russo', 'rating': 8.4, 'title': 'Avengers: Endgame', 'year': 2019}


Nested typeddict

In [36]:
from typing_extensions import TypedDict, Annotated

class Actor(TypedDict):
    name: str
    age: int

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genre: list[str]
    budget: float | None = Field(None, description="Budget of the movie in USD")

model_with_type_dict = model.with_structured_output(MovieDetails)
response = model_with_type_dict.invoke("tell me the details of avengers endgame")
response


{'budget': 356000000,
 'cast': [{'age': 57, 'name': 'Robert Downey Jr.'},
  {'age': 42, 'name': 'Chris Evans'},
  {'age': 58, 'name': 'Mark Ruffalo'},
  {'age': 40, 'name': 'Chris Hemsworth'},
  {'age': 27, 'name': 'Tom Holland'}],
 'genre': ['Action', 'Adventure', 'Science Fiction'],
 'title': 'Avengers: Endgame',
 'year': 2019}

info about the model

In [37]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### DataClasses<break>
a data class is a class typically containing mainly data, altough there arent really any restrictions or validation, you can create it using @dataclass decorator

In [42]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    name: str 
    email:str 
    phone: str 

agent = create_agent(
    model = 'groq:qwen/qwen3-32b',
    response_format = ContactInfo
)

result = agent.invoke({
    "messages":[{"role":"user", "content":"Extract info from: John cenam johncena@gmail.com,908742344"}]
})

result

{'messages': [HumanMessage(content='Extract info from: John cenam johncena@gmail.com,908742344', additional_kwargs={}, response_metadata={}, id='d3faefa3-11af-49db-b167-6b7f63d19634'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract information from the given text: "John cenam johncena@gmail.com,908742344". I need to figure out how to parse this into the ContactInfo function they provided.\n\nFirst, the function requires name, email, and phone. The text has three parts separated by spaces and a comma. Let me break it down. "John cenam" is probably the name. Then "johncena@gmail.com" is the email. The phone number is "908742344". Wait, there\'s a comma between the email and phone number. So the email is "johncena@gmail.com" and the phone is "908742344". \n\nI need to make sure the name is correctly split. "John cenam" might be the full name. Maybe "cenam" is part of the last name? Or maybe there\'s a typo? But the user prov

In [43]:
result['structured_response']

ContactInfo(name='John cenam', email='johncena@gmail.com', phone='908742344')

Doing same thing using pydantic

In [40]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    name: str = Field(description="Name of the person")
    email:str = Field(description="Email of the person")
    phone: str = Field(description="Phone number of the person")

agent = create_agent(
    model = 'groq:qwen/qwen3-32b',
    response_format = ContactInfo
)

result = agent.invoke({
    "messages":[{"role":"user", "content":"Extract info from: John cenam johncena@gmail.com,908742344"}]
})

result

{'messages': [HumanMessage(content='Extract info from: John cenam johncena@gmail.com,908742344', additional_kwargs={}, response_metadata={}, id='e5c4eee5-21a0-4c48-a57c-f6de77081020'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract information from the given text: "John cenam johncena@gmail.com,908742344". First, I need to identify the different pieces of information here. The name is probably "John cenam". Wait, maybe that\'s a typo? Maybe it\'s supposed to be "John Cena", but the user wrote "cenam". Hmm, but I should go with what\'s provided. The email is clearly "johncena@gmail.com". The phone number is "908742344". Let me check the required parameters for the ContactInfo function. The required fields are name, email, and phone. So I need to make sure all three are included. The name here is "John cenam". Even though it might be a typo, I should use that as given. The phone number is a number, so I should format it as 

In [41]:
result['structured_response']

ContactInfo(name='John cenam', email='johncena@gmail.com', phone='908742344')

Doing same thing using typeddict

In [ ]:
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    name: str 
    email:str 
    phone: str 

agent = create_agent(
    model = 'groq:qwen/qwen3-32b',
    response_format = ContactInfo
)

result = agent.invoke({
    "messages":[{"role":"user", "content":"Extract info from: John cenam johncena@gmail.com,908742344"}]
})

result

{'messages': [HumanMessage(content='Extract info from: John cenam johncena@gmail.com,908742344', additional_kwargs={}, response_metadata={}, id='1de631f3-e745-4249-9740-4e391b0f2bc8'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract information from the given text: "John cenam johncena@gmail.com,908742344". First, I need to parse this string to identify the name, email, and phone number.\n\nStarting with the name, the first part is "John cenam". It looks like "cenam" might be a typo for "Cena", considering the email is johncena@gmail.com. So I\'ll correct "cenam" to "Cena" to match the email. That makes the name "John Cena".\n\nNext, the email is clearly "johncena@gmail.com". I just need to confirm that it\'s correctly associated with the name John Cena. \n\nThen the phone number is "908742344". It\'s a numerical string. I should check if it\'s in the correct format. The user might expect it to be formatted as a standard p

In [45]:
result['structured_response']

{'name': 'John Cena', 'email': 'johncena@gmail.com', 'phone': '908742344'}